## **EXPLORACIÓN DEL DATASET DMID_PNG**

Este notebook tiene como objetivo realizar un análisis exploratorio del dataset DMID_PNG (Digital Mammography Dataset for Breast Cancer Diagnosis Research), con el fin de comprender su estructura, calidad y distribución antes de su uso en modelos de deep learning.

En esta fase se lleva a cabo la carga del dataset y la inspección de sus principales componentes: imágenes mamográficas en formato TIFF, máscaras binarias asociadas y anotaciones a nivel de píxel (PLA). Además, se realiza una visualización representativa de ejemplos del conjunto de datos para entender la correspondencia entre imágenes y sus etiquetas. También se incluye un análisis estadístico básico, donde se estudian aspectos como el rango de valores de los píxeles, la distribución de intensidades y la cobertura de las anotaciones. Este análisis permite evaluar la calidad del dataset y detectar posibles problemas como desbalance o ruido en las etiquetas.

Finalmente, se generan diferentes visualizaciones (histogramas, distribuciones y grids de imágenes) que facilitan la interpretación del conjunto de datos y proporcionan una base sólida para las siguientes fases del proyecto, como el preprocesamiento y el entrenamiento de modelos de redes neuronales convolucionales.

In [3]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import re 
import random
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

### **CARGA DEL DATASET**

In [4]:
BASE = Path("..") / "DMID_PNG"   # ajusta si hace falta
print("Dataset path:", BASE.resolve())

Dataset path: C:\Users\aleja\Desktop\3ANYO\redesNeuronales\proyecto\breast-cancer\Bloque2_CNN\DMID_PNG


In [5]:
BASE_1024 = Path("..") / "DMID_PNG" / "1024"

all_files = list(BASE_1024.rglob("*"))

print("USANDO SOLO 1024 (alta resolución)")
print("TOTAL ARCHIVOS:", len(all_files))

print("\nESTRUCTURA 1024 (PRIMERAS DIEZ IMGAENES):")
for f in all_files[:10]:
    print("➡", f)

USANDO SOLO 1024 (alta resolución)
TOTAL ARCHIVOS: 0

ESTRUCTURA 1024 (PRIMERAS DIEZ IMGAENES):


In [6]:
imgs = imgs = [p for p in all_files if p.is_file() and p.suffix.lower() in [".png", ".tif", ".tiff"]]
masks = masks = [p for p in all_files if p.is_file() and "mask" in str(p).lower()]
plas = plas = [p for p in all_files if p.is_file() and "pla" in str(p).lower()]

print("\nRESUMEN 1024:")
print("----------------------------------")
print("Imágenes:", len(imgs))
print("Masks:", len(masks))
print("PLA:", len(plas))
print("----------------------------------")

# gráfico visual
plt.figure(figsize=(6,6))
plt.pie(
    [len(imgs), len(masks), len(plas)],
    labels=["Imágenes (1024)", "Masks (1024)", "PLA (1024)"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Dataset DMID - 1024")
plt.show()


RESUMEN 1024:
----------------------------------
Imágenes: 0
Masks: 0
PLA: 0
----------------------------------


ValueError: cannot convert float NaN to integer

ValueError: need at least one array to concatenate

<Figure size 600x600 with 1 Axes>

In [7]:
meta_path = BASE / "Metadata.xlsx"
raw = pd.read_excel(meta_path, header=None)

df = raw.copy()

df.columns = [
    "image_id",
    "view",
    "tissue",
    "abnormality_type",
    "class",
    "x",
    "y",
    "radius"
]

# solo filas reales
df = df[df["image_id"].astype(str).str.contains("IMG", na=False)].copy()

# limpiar espacios
df["class"] = df["class"].fillna("N").astype(str).str.strip()
df.head(50)

FileNotFoundError: [Errno 2] No such file or directory: '..\\DMID_PNG\\Metadata.xlsx'

In [ ]:
class_counts = df["class"].value_counts()

plt.figure(figsize=(6,6))

plt.pie(
    class_counts,
    labels=class_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Distribución real de clases (1024)")
plt.show()

In [ ]:
def imread(p):
    try:
        return np.array(Image.open(p))
    except:
        return None

### **EMPAREJAR IMÁGENES**

In [ ]:
def extract_id(path):
    """
    Extrae ID tipo IMG001 desde el nombre del archivo
    """
    match = re.search(r'(img\d+)', path.stem.lower())
    return match.group(1) if match else None

In [ ]:
img_dict = {}
mask_dict = {}
pla_dict = {}

# TIFF (base)
for p in imgs:
    k = extract_id(p)
    if k:
        img_dict[k] = p

# MASK
for p in masks:
    k = extract_id(p)
    if k:
        mask_dict[k] = p

# PLA
for p in plas:
    k = extract_id(p)
    if k:
        pla_dict[k] = p

print("- INDEXACIÓN COMPLETADA")
print("----------------------------------")
print(f"  - Imágenes: {len(img_dict)}")
print(f"  - Masks: {len(mask_dict)}")
print(f"  - PLA: {len(pla_dict)}")

In [ ]:
pairs = []
missing_mask = []
missing_pla = []

for k in img_dict.keys():
    img = img_dict.get(k)
    mask = mask_dict.get(k)
    pla = pla_dict.get(k)
    
    if mask and pla:
        pairs.append({
            "id": k,
            "image": img,
            "mask": mask,
            "pla": pla
        })
    else:
        if not mask:
            missing_mask.append(k)
        if not pla:
            missing_pla.append(k)

print("\nEMPAREJAMIENTO FINAL")
print("----------------------------------")
print(f"  - Casos completos (img+mask+pla): {len(pairs)}")
print(f"  - Sin máscara: {len(missing_mask)}")
print(f"  - Sin PLA: {len(missing_pla)}")

In [ ]:
print("\nEjemplos sin mask:", missing_mask[:5])
print("Ejemplos sin PLA:", missing_pla[:5])

In [ ]:
def show_case(p):
    img = np.array(Image.open(p["image"]))
    mask = np.array(Image.open(p["mask"]))
    pla = np.array(Image.open(p["pla"]))

    fig, ax = plt.subplots(1, 4, figsize=(16,4))

    # Imagen original
    ax[0].imshow(img, cmap="gray")
    ax[0].set_title("Imagen")
    ax[0].axis("off")

    # Mask
    ax[1].imshow(mask, cmap="gray")
    ax[1].set_title("Mask")
    ax[1].axis("off")

    # PLA
    ax[2].imshow(pla, cmap="gray")
    ax[2].set_title("PLA")
    ax[2].axis("off")

    # Overlay (MUY IMPORTANTE 🔥)
    ax[3].imshow(img, cmap="gray")
    ax[3].imshow(mask, cmap="jet", alpha=0.4)
    ax[3].set_title("Overlay Mask")
    ax[3].axis("off")

    plt.suptitle(f"Caso: {p['id']}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
print("VISUALIZACIÓN DE CASOS EMPAREJADOS")
print("--------------------------------------------------")

samples = random.sample(pairs, min(5, len(pairs)))

for p in samples:
    show_case(p)

### **ESTADÍSTICAS BÁSICAS**

In [ ]:
pixel_values = []

for p in pairs:
    img = np.array(Image.open(p["image"]))
    pixel_values.append(img.flatten())

pixel_values = np.concatenate(pixel_values)

print("- ESTADÍSTICAS DE INTENSIDAD")
print("----------------------------------")
print(f"  - Min: {pixel_values.min()}")
print(f"  - Max: {pixel_values.max()}")
print(f"  - Mean: {pixel_values.mean():.2f}")
print(f"  - Std: {pixel_values.std():.2f}")

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(pixel_values, bins=256)
plt.title("Distribución de intensidades de píxeles")
plt.xlabel("Intensidad")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
# CUANTO OCUPA LA LESIÓN EN LA IMAGEN
coverages = []

for p in pairs:
    mask = np.array(Image.open(p["mask"]))
    
    # binarizar por si acaso
    mask = (mask > 0).astype(np.uint8)
    
    coverage = mask.mean()   # % de píxeles con lesión
    coverages.append(coverage)

coverages = np.array(coverages)

print("- COBERTURA DE LESIONES")
print("----------------------------------")
print(f"  - Media: {coverages.mean()*100:.2f}%")
print(f"  - Min: {coverages.min()*100:.2f}%")
print(f"  - Max: {coverages.max()*100:.2f}%")

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(coverages * 100, bins=30)
plt.title("Distribución de cobertura de lesiones (%)")
plt.xlabel("% de imagen ocupada")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
sizes = []

for p in pairs:
    img = np.array(Image.open(p["image"]))
    sizes.append(img.shape)

unique_sizes = set(sizes)

print("- TAMAÑOS DE IMAGEN")
print("----------------------------------")
print(unique_sizes)

In [ ]:
# índice de menor y mayor cobertura
min_idx = np.argmin(coverages)
max_idx = np.argmax(coverages)

print("- Ejemplo menor lesión:")
show_case(pairs[min_idx])

print("- Ejemplo mayor lesión:")
show_case(pairs[max_idx])